In [1]:
import pandas as pd

# Load DataSpeciesPerCode
data_path = r"C:\Users\jdelhoyo\PhD\Study cases\Genissiat\RV Characterization\repo-github\data\DefinitiveRV4(withconifers)2.csv"
df_species = pd.read_csv(data_path)

print(f"Loaded DataSpeciesPerCode: {df_species.shape[0]} rows, {df_species.shape[1]} columns")
print(df_species.head())

Loaded DataSpeciesPerCode: 88 rows, 134 columns
   Id_Data  Id_Place  Id_RipUnit  Id_Reach Basin Sub_Basin Stand_Code    Bank  \
0        1         1           1         1  Arve      Arve       A-A1   Left    
1        2         1           2         1  Arve      Arve       A-A1  Right    
2        3         2           3         2  Arve      Arve       A-A2   Left    
3        4         2           4         2  Arve      Arve       A-A2  Right    
4        5         3           5         3  Arve      Arve       A-A3   Left    

      Cod_Plg     RipUnit  ...  sp_11_10-20  sp_11_20-30  sp_11_30-40  \
0   A-A1-Left   A-A1-Left  ...          0.0          0.0          0.0   
1  A-A1-Right  A-A1-Right  ...          0.0          0.0          0.0   
2   A-A2-Left   A-A2-Left  ...          0.0          0.0          0.0   
3  A-A2-Right  A-A2-Right  ...          NaN          NaN          NaN   
4   A-A3-Left   A-A3-Left  ...          0.0          0.0          0.0   

   sp_11_40-50  sp_11_>50 

In [2]:
from pathlib import Path

# Define output directory for results
output_dir = Path(r"C:\Users\jdelhoyo\PhD\Study cases\Genissiat\RV Characterization\repo-github\data\Results\VegetationIndex2")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {output_dir}")
print(f"Directory exists: {output_dir.exists()}")

Output directory: C:\Users\jdelhoyo\PhD\Study cases\Genissiat\RV Characterization\repo-github\data\Results\VegetationIndex2
Directory exists: True


# 1. Data Structure and Exploration

Define species to use and explore the data structure.

In [3]:
# Define valid species codes
valid_species = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 18, 19, 22, 25]

species_names = {
    1: 'Abies',
    2: 'Populus',
    3: 'Salix',
    4: 'Alnus',
    5: 'Fraxinus',
    6: 'Fagus',
    7: 'Sorbus',
    8: 'Quercus',
    9: 'Ulmus',
    10: 'Larix',
    11: 'Pinus',
    12: 'Picea',
    13: 'Prunus',
    15: 'Acer',
    16: 'Castanea',
    18: 'Tilia',
    19: 'Robinia',
    22: 'Aesculus',
    25: 'Juglans'
}

print(f"Valid species: {valid_species}")
print(f"Total valid species: {len(valid_species)}")
print(f"\nData shape: {df_species.shape}")
print(f"Columns in data: {df_species.columns.tolist()[:10]}...")  # Show first 10 columns

Valid species: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 18, 19, 22, 25]
Total valid species: 19

Data shape: (88, 134)
Columns in data: ['Id_Data', 'Id_Place', 'Id_RipUnit', 'Id_Reach', 'Basin', 'Sub_Basin', 'Stand_Code', 'Bank', 'Cod_Plg', 'RipUnit']...


In [4]:
# Identify species presence columns (sp_1, sp_2, ..., sp_29)
species_presence_cols = [col for col in df_species.columns if col.startswith('sp_') and col.split('_')[1].isdigit()]
species_presence_cols.sort(key=lambda x: int(x.split('_')[1]))

# Identify diameter class columns (sp_X_10-20, sp_X_20-30, etc.)
diameter_classes = ['10-20', '20-30', '30-40', '40-50', '>50']
species_diameter_cols = {}

for sp_code in valid_species:
    sp_diameter_cols = []
    for diameter_class in diameter_classes:
        col_name = f'sp_{sp_code}_{diameter_class}'
        if col_name in df_species.columns:
            sp_diameter_cols.append(col_name)
    if sp_diameter_cols:
        species_diameter_cols[sp_code] = sp_diameter_cols

print(f"Species presence columns found: {len(species_presence_cols)}")
print(f"Example: {species_presence_cols[:5]}")
print(f"\nSpecies with diameter class columns: {len(species_diameter_cols)}")
print(f"Example: sp_1 -> {species_diameter_cols.get(1, 'None')}")

# Filter to only valid species presence columns
valid_sp_presence_cols = [f'sp_{sp}' for sp in valid_species if f'sp_{sp}' in species_presence_cols]
print(f"\nValid species presence columns: {len(valid_sp_presence_cols)}")
print(f"Columns: {valid_sp_presence_cols}")

Species presence columns found: 93
Example: ['sp_1', 'sp_1_10-20', 'sp_1_20-30', 'sp_1_30-40', 'sp_1_40-50']

Species with diameter class columns: 19
Example: sp_1 -> ['sp_1_10-20', 'sp_1_20-30', 'sp_1_30-40', 'sp_1_40-50', 'sp_1_>50']

Valid species presence columns: 19
Columns: ['sp_1', 'sp_2', 'sp_3', 'sp_4', 'sp_5', 'sp_6', 'sp_7', 'sp_8', 'sp_9', 'sp_10', 'sp_11', 'sp_12', 'sp_13', 'sp_15', 'sp_16', 'sp_18', 'sp_19', 'sp_22', 'sp_25']


In [5]:
# Prepare analysis dataframe with only valid species
# Keep Cod_Plg column and valid species presence/diameter columns

cols_to_keep = ['Cod_Plg'] if 'Cod_Plg' in df_species.columns else []
cols_to_keep.extend(valid_sp_presence_cols)

# Add diameter class columns
for sp_code in valid_species:
    for diameter_class in diameter_classes:
        col_name = f'sp_{sp_code}_{diameter_class}'
        if col_name in df_species.columns:
            cols_to_keep.append(col_name)

# Create clean dataframe
df_analysis = df_species[cols_to_keep].copy()

print(f"Analysis dataframe shape: {df_analysis.shape}")
print(f"Total columns after filtering: {len(df_analysis.columns)}")
print(f"\nFirst few rows:")
print(df_analysis.head())
print(f"\nColumn summary:")
print(f"  - ID column: Cod_Plg")
print(f"  - Species presence columns: {len(valid_sp_presence_cols)}")
print(f"  - Diameter class columns: {len(cols_to_keep) - len(valid_sp_presence_cols) - 1}")
print(f"  - Total: {len(cols_to_keep)-1} species data columns")

Analysis dataframe shape: (88, 84)
Total columns after filtering: 84

First few rows:
      Cod_Plg  sp_1  sp_2  sp_3  sp_4  sp_5  sp_6  sp_7  sp_8  sp_9  ...  \
0   A-A1-Left     1     0     1     1     1     0     0     0     0  ...   
1  A-A1-Right     0     0     1     1     0     0     1     0     0  ...   
2   A-A2-Left     0     1     1     1     0     0     0     0     0  ...   
3  A-A2-Right     0     1     1     1     0     0     0     0     1  ...   
4   A-A3-Left     0     1     0     0     1     0     0     0     0  ...   

   sp_16_20-30  sp_16_30-40  sp_18_10-20  sp_19_10-20  sp_19_20-30  \
0            0            0            0            0            0   
1            0            0            0            0            0   
2            0            0            0            1            0   
3            0            0            0            1            1   
4            0            0            0            0            0   

   sp_22_10-20  sp_22_30-40  sp_25_1

# 2. Calculate Species Richness Index

Calculate richness metrics from species presence data.

In [6]:
# Step 1: Calculate genus richness (S)
# Count how many valid tree genera are present (value > 0) in each row

df_analysis['S'] = (df_analysis[valid_sp_presence_cols] > 0).sum(axis=1)

print("Step 1: Genus Richness (S)")
print("="*60)
print("S = count of valid tree genera present (sp_X > 0)")
print("\nRichness statistics:")
print(f"  - Min: {df_analysis['S'].min()}")
print(f"  - Max: {df_analysis['S'].max()}")
print(f"  - Mean: {df_analysis['S'].mean():.2f}")
print(f"  - Median: {df_analysis['S'].median():.0f}")
print("\nRichness distribution:")
print(df_analysis['S'].value_counts().sort_index())
print("\nFirst 10 rows:")
print(df_analysis[['Cod_Plg', 'S']].head(10))

Step 1: Genus Richness (S)
S = count of valid tree genera present (sp_X > 0)

Richness statistics:
  - Min: 2
  - Max: 9
  - Mean: 5.33
  - Median: 6

Richness distribution:
S
2     3
3    10
4    11
5    18
6    28
7    14
8     3
9     1
Name: count, dtype: int64

First 10 rows:
      Cod_Plg  S
0   A-A1-Left  7
1  A-A1-Right  5
2   A-A2-Left  4
3  A-A2-Right  6
4   A-A3-Left  3
5  A-A3-Right  3
6   A-A4-Left  4
7  A-A4-Right  3
8   A-A5-Left  8
9  A-A5-Right  7


In [7]:
# Step 2: Calculate maximum genus richness observed
S_max = df_analysis['S'].max()

print("\nStep 2: Maximum Genus Richness Observed")
print("="*60)
print(f"S_max = {S_max}")
print("\nThis is the maximum number of tree genera present in any single margin")

if S_max <= 1:
    raise ValueError(
        "S_max must be greater than 1 to calculate Jsp = ln(S) / ln(S_max). "
        "Check genus richness values."
    )


Step 2: Maximum Genus Richness Observed
S_max = 9

This is the maximum number of tree genera present in any single margin


In [8]:
# Step 3: Calculate logarithmic transformation of genus richness
# lnS = ln(S) if S > 0, else 0

import numpy as np

df_analysis['lnS'] = df_analysis['S'].apply(lambda x: np.log(x) if x > 0 else 0)

print("\nStep 3: Logarithmic Transformation of Genus Richness (lnS)")
print("="*60)
print("lnS = ln(S) if S > 0, else 0")
print("\nlnS statistics:")
print(f"  - Min: {df_analysis['lnS'].min():.4f}")
print(f"  - Max: {df_analysis['lnS'].max():.4f}")
print(f"  - Mean: {df_analysis['lnS'].mean():.4f}")
print(f"  - Median: {df_analysis['lnS'].median():.4f}")
print("\nFirst 10 rows:")
print(df_analysis[['Cod_Plg', 'S', 'lnS']].head(10))


Step 3: Logarithmic Transformation of Genus Richness (lnS)
lnS = ln(S) if S > 0, else 0

lnS statistics:
  - Min: 0.6931
  - Max: 2.1972
  - Mean: 1.6265
  - Median: 1.7918

First 10 rows:
      Cod_Plg  S       lnS
0   A-A1-Left  7  1.945910
1  A-A1-Right  5  1.609438
2   A-A2-Left  4  1.386294
3  A-A2-Right  6  1.791759
4   A-A3-Left  3  1.098612
5  A-A3-Right  3  1.098612
6   A-A4-Left  4  1.386294
7  A-A4-Right  3  1.098612
8   A-A5-Left  8  2.079442
9  A-A5-Right  7  1.945910


In [9]:
# Step 4: Calculate standardized genus richness component (Jsp)
# Jsp = ln(S) / ln(S_max), bounded between 0 and 1

def calculate_jsp(row, s_max):
    s = row['S']
    ln_s = row['lnS']
    
    if s_max <= 1:
        # If maximum richness is 0 or 1, use a binary standardization
        return 1 if s > 0 else 0
    else:
        # Standard case: normalize by ln(S_max)
        return ln_s / np.log(s_max)

df_analysis['Jsp'] = df_analysis.apply(lambda row: calculate_jsp(row, S_max), axis=1)

print("\nStep 4: Standardized Genus Richness Component (Jsp)")
print("="*60)
print("Normalization formula:")

if S_max <= 1:
    print(f"  - S_max = {S_max} (≤ 1): Jsp = 1 if S > 0, else 0")
else:
    print(f"  - S_max = {S_max} (> 1): Jsp = lnS / ln({S_max})")
    print(f"  - Normalization factor: ln({S_max}) = {np.log(S_max):.4f}")

print("\nStandardized richness statistics:")
print(f"  - Min: {df_analysis['Jsp'].min():.4f}")
print(f"  - Max: {df_analysis['Jsp'].max():.4f}")
print(f"  - Mean: {df_analysis['Jsp'].mean():.4f}")
print(f"  - Median: {df_analysis['Jsp'].median():.4f}")

print("\nFull richness summary (first 15 rows):")
summary_cols = ['Cod_Plg', 'S', 'lnS', 'Jsp']
print(df_analysis[summary_cols].head(15).to_string(index=False))

print("\n✓ Genus richness component calculated and standardized between 0 and 1")


Step 4: Standardized Genus Richness Component (Jsp)
Normalization formula:
  - S_max = 9 (> 1): Jsp = lnS / ln(9)
  - Normalization factor: ln(9) = 2.1972

Standardized richness statistics:
  - Min: 0.3155
  - Max: 1.0000
  - Mean: 0.7403
  - Median: 0.8155

Full richness summary (first 15 rows):
   Cod_Plg  S      lnS      Jsp
 A-A1-Left  7 1.945910 0.885622
A-A1-Right  5 1.609438 0.732487
 A-A2-Left  4 1.386294 0.630930
A-A2-Right  6 1.791759 0.815465
 A-A3-Left  3 1.098612 0.500000
A-A3-Right  3 1.098612 0.500000
 A-A4-Left  4 1.386294 0.630930
A-A4-Right  3 1.098612 0.500000
 A-A5-Left  8 2.079442 0.946395
A-A5-Right  7 1.945910 0.885622
 A-A6-Left  7 1.945910 0.885622
A-A6-Right  5 1.609438 0.732487
 A-A7-Left  6 1.791759 0.815465
A-A7-Right  9 2.197225 1.000000
 A-A8-Left  8 2.079442 0.946395

✓ Genus richness component calculated and standardized between 0 and 1


In [10]:
# Export genus richness component
richness_export = df_analysis[['Cod_Plg', 'S', 'lnS', 'Jsp']].copy()

richness_file = output_dir / 'GenusRichnessComponent.csv'
richness_export.to_csv(richness_file, index=False)

print(f"\n✓ Exported genus richness component to: {richness_file.name}")


✓ Exported genus richness component to: GenusRichnessComponent.csv


# 3. Calculate Diameter Diversity Index

Calculate Shannon diversity and equitability across diameter classes.

In [11]:
# Step 1: Calculate number of genera per diameter class
# For each row and each diameter class, count valid genera with presence > 0

diameter_classes_list = ['10-20', '20-30', '30-40', '40-50', '>50']

for diam_class in diameter_classes_list:
    # Get all sp_X_diameter_class columns for valid genera
    cols_for_class = []
    for sp_code in valid_species:
        col_name = f'sp_{sp_code}_{diam_class}'
        if col_name in df_analysis.columns:
            cols_for_class.append(col_name)
    
    # Count genera present in this diameter class
    col_name_n = f'n_{diam_class}'
    df_analysis[col_name_n] = (df_analysis[cols_for_class] > 0).sum(axis=1)

print("Step 1: Genus count per diameter class")
print("="*60)
print("For each row, counted valid tree genera present in each diameter class.")
print("\nColumns added: n_10-20, n_20-30, n_30-40, n_40-50, n_>50")
print("\nExample, first 10 rows:")
count_cols = ['Cod_Plg'] + [f'n_{dc}' for dc in diameter_classes_list]
print(df_analysis[count_cols].head(10).to_string(index=False))

Step 1: Genus count per diameter class
For each row, counted valid tree genera present in each diameter class.

Columns added: n_10-20, n_20-30, n_30-40, n_40-50, n_>50

Example, first 10 rows:
   Cod_Plg  n_10-20  n_20-30  n_30-40  n_40-50  n_>50
 A-A1-Left        4        4        1        1      0
A-A1-Right        2        2        2        1      1
 A-A2-Left        4        2        1        0      0
A-A2-Right        4        4        0        1      0
 A-A3-Left        3        1        0        1      0
A-A3-Right        2        2        1        1      0
 A-A4-Left        4        2        1        0      0
A-A4-Right        3        2        0        1      0
 A-A5-Left        5        6        1        1      0
A-A5-Right        4        5        2        0      0


In [12]:
# Step 2: Calculate diameter structure component
# This includes:
#   C      = number of occupied diameter classes
#   J_C    = standardized diameter class richness
#   H_dbh  = Shannon diversity across occupied diameter classes
#   J_dbh  = Pielou evenness across occupied diameter classes
#   J_diam = corrected diameter structure component = J_C * J_dbh

C_max = len(diameter_classes_list)

def calculate_diameter_structure(row):
    """
    Calculate diameter structure metrics across DBH classes.

    H_dbh is Shannon diversity based on the distribution of genus occurrences
    across occupied DBH classes.

    J_dbh is Pielou evenness among occupied DBH classes.

    J_C represents the proportion of DBH classes occupied.

    J_diam combines DBH class occupation and evenness, so high values require
    both several occupied DBH classes and an even distribution among them.
    """
    
    counts = [
        row['n_10-20'],
        row['n_20-30'],
        row['n_30-40'],
        row['n_40-50'],
        row['n_>50']
    ]
    
    # Number of occupied diameter classes
    active = [c for c in counts if c > 0]
    C = len(active)
    
    # Standardized diameter class occupation
    J_C = C / C_max if C_max > 0 else 0
    
    if C == 0:
        # No diameter class represented
        H_dbh = 0
        J_dbh = 0
        J_diam = 0
    
    elif C == 1:
        # Only one diameter class represented.
        # No among-class diameter diversity is present.
        H_dbh = 0
        J_dbh = 0
        J_diam = 0
    
    else:
        # C >= 2: calculate Shannon diversity and Pielou evenness
        total = sum(active)
        proportions = [a / total for a in active]
        
        H_dbh = -sum(p * np.log(p) for p in proportions if p > 0)
        J_dbh = H_dbh / np.log(C)
        
        # Corrected diameter component
        J_diam = J_C * J_dbh
    
    return C, J_C, H_dbh, J_dbh, J_diam


# Apply calculation to each row
results = df_analysis.apply(calculate_diameter_structure, axis=1, result_type='expand')

df_analysis['C_dbh'] = results[0]
df_analysis['J_C'] = results[1]
df_analysis['H_dbh'] = results[2]
df_analysis['J_dbh'] = results[3]
df_analysis['J_diam'] = results[4]


print("\nStep 2: Diameter Structure Component")
print("="*60)
print("Formula:")
print("  - C_dbh = number of occupied DBH classes")
print(f"  - C_max = {C_max}")
print("  - J_C = C_dbh / C_max")
print("  - If C_dbh = 0: H_dbh = 0, J_dbh = 0, J_diam = 0")
print("  - If C_dbh = 1: H_dbh = 0, J_dbh = 0, J_diam = 0")
print("  - If C_dbh ≥ 2: H_dbh = -Σ(p_i * ln(p_i)), J_dbh = H_dbh / ln(C_dbh)")
print("  - J_diam = J_C * J_dbh")

print("\nStatistics:")
print(f"  C_dbh  - Min: {df_analysis['C_dbh'].min():.0f}, Max: {df_analysis['C_dbh'].max():.0f}, Mean: {df_analysis['C_dbh'].mean():.2f}")
print(f"  J_C    - Min: {df_analysis['J_C'].min():.4f}, Max: {df_analysis['J_C'].max():.4f}, Mean: {df_analysis['J_C'].mean():.4f}")
print(f"  H_dbh  - Min: {df_analysis['H_dbh'].min():.4f}, Max: {df_analysis['H_dbh'].max():.4f}, Mean: {df_analysis['H_dbh'].mean():.4f}")
print(f"  J_dbh  - Min: {df_analysis['J_dbh'].min():.4f}, Max: {df_analysis['J_dbh'].max():.4f}, Mean: {df_analysis['J_dbh'].mean():.4f}")
print(f"  J_diam - Min: {df_analysis['J_diam'].min():.4f}, Max: {df_analysis['J_diam'].max():.4f}, Mean: {df_analysis['J_diam'].mean():.4f}")

print("\nDistribution of occupied DBH classes:")
print(df_analysis['C_dbh'].value_counts().sort_index())

print("\nExample, first 15 rows:")
example_cols = (
    ['Cod_Plg'] +
    [f'n_{dc}' for dc in diameter_classes_list] +
    ['C_dbh', 'J_C', 'H_dbh', 'J_dbh', 'J_diam']
)
print(df_analysis[example_cols].head(15).to_string(index=False))

print("\n✓ Diameter structure component calculated")


Step 2: Diameter Structure Component
Formula:
  - C_dbh = number of occupied DBH classes
  - C_max = 5
  - J_C = C_dbh / C_max
  - If C_dbh = 0: H_dbh = 0, J_dbh = 0, J_diam = 0
  - If C_dbh = 1: H_dbh = 0, J_dbh = 0, J_diam = 0
  - If C_dbh ≥ 2: H_dbh = -Σ(p_i * ln(p_i)), J_dbh = H_dbh / ln(C_dbh)
  - J_diam = J_C * J_dbh

Statistics:
  C_dbh  - Min: 2, Max: 5, Mean: 3.32
  J_C    - Min: 0.4000, Max: 1.0000, Mean: 0.6636
  H_dbh  - Min: 0.5004, Max: 1.5596, Mean: 1.0477
  J_dbh  - Min: 0.7219, Max: 1.0000, Mean: 0.8989
  J_diam - Min: 0.2888, Max: 0.9690, Mean: 0.5940

Distribution of occupied DBH classes:
C_dbh
2.0    13
3.0    37
4.0    35
5.0     3
Name: count, dtype: int64

Example, first 15 rows:
   Cod_Plg  n_10-20  n_20-30  n_30-40  n_40-50  n_>50  C_dbh  J_C    H_dbh    J_dbh   J_diam
 A-A1-Left        4        4        1        1      0    4.0  0.8 1.193550 0.860964 0.688771
A-A1-Right        2        2        2        1      1    5.0  1.0 1.559581 0.969022 0.969022
 A-A2-Le

In [13]:
# Export diameter structure results
diameter_export_cols = (
    ['Cod_Plg'] +
    [f'n_{dc}' for dc in diameter_classes_list] +
    ['C_dbh', 'J_C', 'H_dbh', 'J_dbh', 'J_diam']
)

diameter_export = df_analysis[diameter_export_cols].copy()

diameter_file = output_dir / 'DiameterStructureComponent.csv'
diameter_export.to_csv(diameter_file, index=False)

print(f"\n✓ Exported diameter structure component to: {diameter_file.name}")


✓ Exported diameter structure component to: DiameterStructureComponent.csv


In [14]:
# Step 3: Check diameter structure calculations

print("\nStep 3: Check diameter structure calculations")
print("="*60)

print(f"Rows with C_dbh = 0:  {len(df_analysis[df_analysis['C_dbh'] == 0])}")
print(f"Rows with C_dbh = 1:  {len(df_analysis[df_analysis['C_dbh'] == 1])}")
print(f"Rows with C_dbh >= 2: {len(df_analysis[df_analysis['C_dbh'] >= 2])}")

print("\nNaN check:")
print(f"  H_dbh:  {df_analysis['H_dbh'].isna().sum()}")
print(f"  J_dbh:  {df_analysis['J_dbh'].isna().sum()}")
print(f"  J_C:    {df_analysis['J_C'].isna().sum()}")
print(f"  J_diam: {df_analysis['J_diam'].isna().sum()}")

print("\nRange check:")
print(f"  J_C    - Min: {df_analysis['J_C'].min():.4f}, Max: {df_analysis['J_C'].max():.4f}")
print(f"  J_dbh  - Min: {df_analysis['J_dbh'].min():.4f}, Max: {df_analysis['J_dbh'].max():.4f}")
print(f"  J_diam - Min: {df_analysis['J_diam'].min():.4f}, Max: {df_analysis['J_diam'].max():.4f}")

print("\n✓ Diameter structure calculations checked")


Step 3: Check diameter structure calculations
Rows with C_dbh = 0:  0
Rows with C_dbh = 1:  0
Rows with C_dbh >= 2: 88

NaN check:
  H_dbh:  0
  J_dbh:  0
  J_C:    0
  J_diam: 0

Range check:
  J_C    - Min: 0.4000, Max: 1.0000
  J_dbh  - Min: 0.7219, Max: 1.0000
  J_diam - Min: 0.2888, Max: 0.9690

✓ Diameter structure calculations checked


In [15]:
# Step 4: Calculate revised combined structural index
# StructuralIndex = 0.5 * Jsp + 0.5 * J_diam

alpha = 0.5

df_analysis['StructuralIndex'] = (
    alpha * df_analysis['Jsp'] +
    (1 - alpha) * df_analysis['J_diam']
)

print("\nStep 4: Revised Combined Structural Index (StructuralIndex)")
print("="*60)
print(f"Formula: StructuralIndex = {alpha} * Jsp + {1-alpha} * J_diam")
print("\nComponents:")
print(f"  - Standardized genus richness, Jsp: {alpha*100:.0f}%")
print(f"  - Corrected diameter structure, J_diam: {(1-alpha)*100:.0f}%")

print("\nStatistics:")
print(f"  Min: {df_analysis['StructuralIndex'].min():.4f}")
print(f"  Max: {df_analysis['StructuralIndex'].max():.4f}")
print(f"  Mean: {df_analysis['StructuralIndex'].mean():.4f}")
print(f"  Median: {df_analysis['StructuralIndex'].median():.4f}")

print("\nComponent statistics:")
print(f"  Jsp    - Min: {df_analysis['Jsp'].min():.4f}, Max: {df_analysis['Jsp'].max():.4f}, Mean: {df_analysis['Jsp'].mean():.4f}")
print(f"  J_diam - Min: {df_analysis['J_diam'].min():.4f}, Max: {df_analysis['J_diam'].max():.4f}, Mean: {df_analysis['J_diam'].mean():.4f}")

print("\nExample, first 15 rows:")
summary_cols = ['Cod_Plg', 'S', 'Jsp', 'C_dbh', 'J_C', 'J_dbh', 'J_diam', 'StructuralIndex']
print(df_analysis[summary_cols].head(15).to_string(index=False))

print("\n✓ Revised structural index calculated")
print(f"\nExample (first 10 rows):")
example_cols = ['Cod_Plg', 'J_sp_proxy', 'J_dbh', 'D_comb']
print(df_analysis[example_cols].head(10).to_string(index=False))


Step 4: Revised Combined Structural Index (StructuralIndex)
Formula: StructuralIndex = 0.5 * Jsp + 0.5 * J_diam

Components:
  - Standardized genus richness, Jsp: 50%
  - Corrected diameter structure, J_diam: 50%

Statistics:
  Min: 0.3414
  Max: 0.8508
  Mean: 0.6671
  Median: 0.6687

Component statistics:
  Jsp    - Min: 0.3155, Max: 1.0000, Mean: 0.7403
  J_diam - Min: 0.2888, Max: 0.9690, Mean: 0.5940

Example, first 15 rows:
   Cod_Plg  S      Jsp  C_dbh  J_C    J_dbh   J_diam  StructuralIndex
 A-A1-Left  7 0.885622    4.0  0.8 0.860964 0.688771         0.787197
A-A1-Right  5 0.732487    5.0  1.0 0.969022 0.969022         0.850755
 A-A2-Left  4 0.630930    3.0  0.6 0.869916 0.521949         0.576440
A-A2-Right  6 0.815465    3.0  0.6 0.878347 0.527008         0.671237
 A-A3-Left  3 0.500000    3.0  0.6 0.864974 0.518984         0.509492
A-A3-Right  3 0.500000    4.0  0.8 0.959148 0.767318         0.633659
 A-A4-Left  4 0.630930    3.0  0.6 0.869916 0.521949         0.576440
A-A4-

KeyError: "['J_sp_proxy', 'D_comb'] not in index"

In [16]:
# Step 5: Calculate percentiles and classify the revised structural index
# Create qualitative classes based on StructuralIndex quartiles

p25 = df_analysis['StructuralIndex'].quantile(0.25)
p50 = df_analysis['StructuralIndex'].quantile(0.50)
p75 = df_analysis['StructuralIndex'].quantile(0.75)

def classify_structure(value):
    if value < p25:
        return 'Low'
    elif value < p50:
        return 'Medium'
    elif value < p75:
        return 'High'
    else:
        return 'Very high'

df_analysis['StructuralClass'] = df_analysis['StructuralIndex'].apply(classify_structure)

print("\nStep 5: Qualitative Classification of Structural Index")
print("="*60)
print("Percentiles:")
print(f"  - p25 (25%): {p25:.4f}")
print(f"  - p50 (50%): {p50:.4f}")
print(f"  - p75 (75%): {p75:.4f}")

print("\nClasses:")
print(f"  'Low':       StructuralIndex < {p25:.4f}")
print(f"  'Medium':    {p25:.4f} <= StructuralIndex < {p50:.4f}")
print(f"  'High':      {p50:.4f} <= StructuralIndex < {p75:.4f}")
print(f"  'Very high': StructuralIndex >= {p75:.4f}")

print("\nClass distribution:")
print(df_analysis['StructuralClass'].value_counts().sort_index())

print("\nExample, first 15 rows:")
example_cols = ['Cod_Plg', 'StructuralIndex', 'StructuralClass']
print(df_analysis[example_cols].head(15).to_string(index=False))

print("\n✓ Structural classification completed")


Step 5: Qualitative Classification of Structural Index
Percentiles:
  - p25 (25%): 0.5912
  - p50 (50%): 0.6687
  - p75 (75%): 0.7457

Classes:
  'Low':       StructuralIndex < 0.5912
  'Medium':    0.5912 <= StructuralIndex < 0.6687
  'High':      0.6687 <= StructuralIndex < 0.7457
  'Very high': StructuralIndex >= 0.7457

Class distribution:
StructuralClass
High         23
Low          22
Medium       21
Very high    22
Name: count, dtype: int64

Example, first 15 rows:
   Cod_Plg  StructuralIndex StructuralClass
 A-A1-Left         0.787197       Very high
A-A1-Right         0.850755       Very high
 A-A2-Left         0.576440             Low
A-A2-Right         0.671237            High
 A-A3-Left         0.509492             Low
A-A3-Right         0.633659          Medium
 A-A4-Left         0.576440             Low
A-A4-Right         0.526186             Low
 A-A5-Left         0.796063       Very high
A-A5-Right         0.725768            High
 A-A6-Left         0.633698          M

In [17]:
# Final export: Complete revised structural index results

final_output_cols = [
    'Cod_Plg',
    'S',
    'lnS',
    'Jsp',
    'C_dbh',
    'J_C',
    'H_dbh',
    'J_dbh',
    'J_diam',
    'StructuralIndex',
    'StructuralClass'
]

df_final = df_analysis[final_output_cols].copy()

# Export to CSV
final_file = output_dir / 'VegetationStructuralIndex_Revised.csv'
df_final.to_csv(final_file, index=False)

print("\n" + "="*60)
print("FINAL REVISED STRUCTURAL INDEX RESULTS EXPORTED")
print("="*60)
print(f"\nFile: {final_file.name}")
print(f"Rows: {len(df_final):,}")
print(f"Columns: {len(df_final.columns)}")

print("\nColumn structure:")
for i, col in enumerate(final_output_cols, 1):
    print(f"  {i}. {col}")

print("\nSummary statistics:")
print(df_final.describe().round(4))

print("\nClass distribution (StructuralClass):")
print(df_final['StructuralClass'].value_counts().sort_index())

print("\n✓ Complete revised structural index calculated and exported")
print(f"✓ File saved: {final_file}")


FINAL REVISED STRUCTURAL INDEX RESULTS EXPORTED

File: VegetationStructuralIndex_Revised.csv
Rows: 88
Columns: 11

Column structure:
  1. Cod_Plg
  2. S
  3. lnS
  4. Jsp
  5. C_dbh
  6. J_C
  7. H_dbh
  8. J_dbh
  9. J_diam
  10. StructuralIndex
  11. StructuralClass

Summary statistics:
             S      lnS      Jsp    C_dbh      J_C    H_dbh    J_dbh   J_diam  \
count  88.0000  88.0000  88.0000  88.0000  88.0000  88.0000  88.0000  88.0000   
mean    5.3295   1.6265   0.7403   3.3182   0.6636   1.0477   0.8989   0.5940   
std     1.4988   0.3247   0.1478   0.7663   0.1533   0.2173   0.0613   0.1342   
min     2.0000   0.6931   0.3155   2.0000   0.4000   0.5004   0.7219   0.2888   
25%     4.0000   1.3863   0.6309   3.0000   0.6000   0.9557   0.8644   0.5219   
50%     6.0000   1.7918   0.8155   3.0000   0.6000   1.0579   0.9128   0.5778   
75%     6.0000   1.7918   0.8155   4.0000   0.8000   1.2135   0.9480   0.7003   
max     9.0000   2.1972   1.0000   5.0000   1.0000   1.5596  

In [18]:
# Class count summary

print("\n" + "="*60)
print("STRUCTURAL CLASS DISTRIBUTION SUMMARY")
print("="*60)

# Debug: check unique values in the class column
print("\nUnique values in StructuralClass column:")
print(df_final['StructuralClass'].unique())

print("\nValue counts, unsorted:")
print(df_final['StructuralClass'].value_counts())

class_counts = df_final['StructuralClass'].value_counts()
total_records = len(df_final)

print(f"\nTotal records: {total_records:,}\n")

# Sort in order: Low, Medium, High, Very high
class_order = ['Low', 'Medium', 'High', 'Very high']

for class_name in class_order:
    if class_name in class_counts.index:
        count = class_counts[class_name]
        percentage = (count / total_records) * 100
        print(f"  {class_name:12} : {count:6,} margins  ({percentage:6.2f}%)")
    else:
        print(f"  {class_name:12} :      0 margins  (  0.00%)")

print("\n✓ Structural class count completed")


STRUCTURAL CLASS DISTRIBUTION SUMMARY

Unique values in StructuralClass column:
['Very high' 'Low' 'High' 'Medium']

Value counts, unsorted:
StructuralClass
High         23
Very high    22
Low          22
Medium       21
Name: count, dtype: int64

Total records: 88

  Low          :     22 margins  ( 25.00%)
  Medium       :     21 margins  ( 23.86%)
  High         :     23 margins  ( 26.14%)
  Very high    :     22 margins  ( 25.00%)

✓ Structural class count completed
